# 04 - Damage and cut-off analysis

Run first: `make baseline` (or `make infer`), then `make damage cutoff flowpath`. For a quick look without satellite data: `python -m src.demo_data` and point `OUT` below at `../data/demo/outputs` (synthetic, clearly labelled).

Attribution: Contains modified Copernicus Sentinel data 2026. (c) OpenStreetMap contributors (ODbL), pre-event snapshot.

In [ ]:
import json
from pathlib import Path
import geopandas as gpd, pandas as pd
from src.facts import Facts

OUT = Path('../data/outputs')   # use '../data/demo/outputs' for the synthetic demo
facts = Facts.load(OUT)
print('missing:', facts.missing(), '| synthetic:', facts.event['synthetic'])
for name in ('flood_extent', 'summarize_damage', 'list_cutoff_settlements', 'settlements_along_flowpath'):
    print(name, json.dumps(getattr(facts, name)(), indent=1, ensure_ascii=False)[:1200])

## Buildings, roads, bridges in the hazard zone
`exposed` means inside the flood/debris footprint at 10 m, not collapsed. OSM completeness in rural Nepal is partial, so counts are lower bounds.

In [ ]:
bld = gpd.read_file(OUT / 'damage.gpkg', layer='buildings_hit')
display(bld.drop(columns='geometry').groupby(['severity', 'hazard']).size().unstack(fill_value=0))
roads = gpd.read_file(OUT / 'damage.gpkg', layer='roads_hit')
display(roads.groupby('highway')['length_hit_m'].sum().div(1000).round(2).rename('km in hazard'))

## Cut-off settlements
Two scenarios because a 10 m mask cannot tell whether a bridge survived: **optimistic** (bridges hold) and **conservative** (exposed bridges fail). `unreachable_pre` / `no_road_access` mean OSM has no route even before the flood, so they are *not* counted as victims of it.

In [ ]:
df = pd.read_csv(OUT / 'cutoff_settlements.csv')
display(df[['name', 'kind', 'risk', 'buildings_300m', 'pre_dist_km', 'post_dist_km_optimistic', 'post_dist_km_conservative']])
print(json.dumps(json.loads((OUT / 'cutoff_summary.json').read_text())['qa'], indent=1))

## Graph validation (the `qa` block above)
Check that `graph_components_pre` is small (a fragmented graph means OSM gaps, not flood damage), that most settlements snap to a road, and that `edges_blocked_*` is plausible for the mapped hazard area.

In [ ]:
from src.report import render_map_png
from src.damage import load_hazard
flood, debris, grid, src = load_hazard(OUT, 'auto', reports=Path('../reports'))
osm = next(p for p in (OUT.parent / 'raw/osm/osm_pre_event.gpkg', OUT.parent / 'osm/osm_pre_event.gpkg') if p.exists())
from IPython.display import Image
Image(render_map_png(OUT, osm, flood, debris, grid))